# CR-Zero: Gameplay Extraction & Policy Training Pipeline

This notebook extracts `(State, Action, Reward)` trajectories from Clash Royale YouTube VoDs and trains the **Deck-Conditioned Policy Network** using Offline Imitation Learning (Behavior Cloning).

It works seamlessly in **both environments**:
1. **Local IDE (VS Code / Antigravity)**: Uses your local Python environment and local GPU (NVIDIA RTX 3050).
2. **Google Colab**: Leverages free T4 / A100 GPUs.

In [ ]:
# Cell 1: Environment & Dependency Setup
import os
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("[*] Running in Google Colab environment")
    !apt-get update -qq && apt-get install -y -qq ffmpeg git
    !pip install -q yt-dlp ultralytics onnx onnxruntime-gpu opencv-python
    if not os.path.exists("clash-ml-model"):
        !git clone https://github.com/varun-zero/clash-ml-model.git
        %cd clash-ml-model
else:
    print("[*] Running in Local IDE environment")

# Add repository root to python search path
repo_root = os.path.abspath(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device:   {torch.cuda.get_device_name(0)}")


## 1. Extract Trajectories from YouTube Gameplay

We download high-level gameplay VoDs (e.g. CRL World Finals, Top 200 Ladder replays), sample frames at 10 FPS, detect deployment clocks and card replacements, and record `(State_{t-0.5s}, Action_t)` pairs canonicalized so ally is always at the bottom.

In [ ]:
# Cell 2: Extract Trajectories from YouTube VoDs
import os
from perception.detector import ClashRoyaleDetector
from colab_pipeline.yt_action_extractor import YouTubeTrajectoryExtractor

# Configure VoD URLs and the player perspective ('bottom' or 'top')
# The extractor canonicalizes 'top' perspective by rotating 180° so ally is always bottom
YOUTUBE_URLS = [
    # Replace with any Clash Royale gameplay YouTube video URL:
    # ("https://www.youtube.com/watch?v=EXAMPLE_ID", "bottom"),
]

weights_path = "checkpoints/yolov8_cr_units.pt" if os.path.exists("checkpoints/yolov8_cr_units.pt") else None
detector = ClashRoyaleDetector(weights=weights_path)

extractor = YouTubeTrajectoryExtractor(
    detector=detector,
    lookback_frames=5,   # 500ms lookback before card leaves hand
    lookahead_frames=2,  # 200ms forward lookahead
    wait_ratio=0.5       # 1 Wait sample for every 2 Card plays (~33% waits in dataset)
)

os.makedirs("./data/trajectories", exist_ok=True)

if not YOUTUBE_URLS:
    print("[!] YOUTUBE_URLS list is currently empty. Add YouTube video URLs above to extract real shards.")
else:
    for i, (url, player_side) in enumerate(YOUTUBE_URLS):
        out_shard = f"./data/trajectories/shard_{i:03d}.npz"
        print(f"\n--- Extracting [{i+1}/{len(YOUTUBE_URLS)}] (Side: {player_side}): {url} ---")
        extractor.process_video(
            video_source=url,
            player_side=player_side,
            output_npz_path=out_shard,
            max_duration_seconds=360
        )


## 2. Train Behavior Cloning Policy Network

Trains `DeckConditionedPolicy` with:
- **32-channel stacked arena CNN** (last 4 frames at 10 Hz).
- **Summed Card Embeddings**: Learned `Embedding(135, 16)` + Linear Static Features (air/ground, speed, range, building/spell, cost).
- **Factored Autoregressive Heads**: Slot Selection (0=Wait, 1..4=Slots) + Legality-Masked Tile Placement (576 tiles).
- **Outcome-Weighted Loss**: Wins (1.5x) > Draws (1.0x) > Losses (0.5x).

In [ ]:
# Cell 3: Train Policy Network
!python colab_pipeline/train_bc.py \
    --data_dir ./data/trajectories \
    --save_dir ./checkpoints \
    --epochs 25 \
    --batch_size 64 \
    --lr 1e-3


## 3. Export Checkpoints & ONNX Model

Exports the best policy checkpoint to FP16 / FP32 ONNX for high-throughput live inference on the local Windows RTX 3050 GPU.

In [ ]:
# Cell 4: Download Checkpoints (Colab Only)
if IS_COLAB:
    from google.colab import files
    if os.path.exists("./checkpoints/cr_zero_policy_best.pt"):
        files.download("./checkpoints/cr_zero_policy_best.pt")
    if os.path.exists("./checkpoints/cr_zero_policy_best.onnx"):
        files.download("./checkpoints/cr_zero_policy_best.onnx")
else:
    print("[*] Running locally. Model checkpoints are already saved in ./checkpoints/")
